# OpenAI Python SDK

Use the official OpenAI client for Chat Completions, then apply the same patterns with LangChain (`ChatOpenAI`, `ChatPromptTemplate | llm`) for reusable prompts and structured output.


## 1. Overview

This document covers:

- Bootstrapping `OpenAI()` with `python-dotenv` from the project root
- Chat Completions: `messages`, roles, `model`, `temperature`, `max_tokens`
- Reading `choices`, `finish_reason`, `usage`, and `id` from responses
- Typed exception handling for authentication, rate limits, and connectivity
- Optional streaming for incremental output
- Bridging to LangChain: `ChatOpenAI`, `ChatPromptTemplate`, LCEL chains, and structured output


## 2. Motivation

Most application code talks to OpenAI through the **Python SDK**, not raw HTTP. Consistent patterns for loading secrets, building `messages`, parsing responses, and surfacing errors keep services and scripts debuggable.

**LangChain** (`ChatOpenAI` + `ChatPromptTemplate | llm`) calls the same Chat Completions API under the hood. Showing both layers clarifies how raw SDK responses relate to LCEL chains without treating them as competing approaches.

Failures that look like model quality issues are often a wrong kernel, missing `.env`, placeholder keys, or an unhandled `RateLimitError`. Standardize client bootstrap once and reuse it everywhere.


## 3. Concepts

### 3.1 Glossary

| Term | Meaning |
|------|--------|
| **`OpenAI` client** | Official SDK entry point; reads `OPENAI_API_KEY` from the environment |
| **Chat Completions** | `client.chat.completions.create(...)` — multi-turn chat interface |
| **`messages`** | List of `{role, content}` dicts: `system`, `user`, `assistant` |
| **`ChatOpenAI`** | LangChain chat model wrapping the same OpenAI API |
| **`ChatPromptTemplate`** | Role-tagged prompt with `{variables}` filled at invoke time |
| **LCEL chain** | `prompt \| llm` (optionally `\| llm.with_structured_output(...)`) |
| **`finish_reason` / `usage`** | SDK metadata: why generation stopped and token counts |
| **Streaming** | `stream=True` (SDK) or `chain.stream(...)` (LangChain) |

### 3.2 How it works

**SDK path**

1. `load_dotenv(root / ".env")` injects secrets into `os.environ`.
2. `OpenAI()` constructs a client that attaches the API key to each request.
3. You pass `messages` and parameters to `chat.completions.create`.
4. Read `response.choices[0].message.content` for the assistant text.

**LangChain path (same API, higher-level surface)**

1. Same `.env` load; build `ChatOpenAI(model=..., temperature=...)`.
2. Define `ChatPromptTemplate.from_messages([...])` with `{vars}`.
3. Compose `chain = prompt | llm` and call `.invoke({...})` / `.stream({...})`.
4. Read `AIMessage.content`, or a Pydantic model via `with_structured_output`.

### 3.3 When to use which

**Use the SDK when:** you need full control over response metadata (`usage`, `finish_reason`, raw streaming deltas) or a minimal dependency surface.

**Use LangChain when:** you want reusable prompts, LCEL composition, batching, and structured output for application workflows.

**Trade-offs:** the SDK is thin and explicit; LangChain adds abstractions and multi-provider portability.


## 4. Architecture

Both paths share `.env` and the OpenAI REST API. LangChain sits on top of the same Chat Completions surface.

### 4.1 End-to-end data flow

```mermaid
flowchart TB
    subgraph setup["Runtime setup"]
        env[".env<br/>OPENAI_API_KEY"]
        env --> sdk["OpenAI client"]
        env --> llm["ChatOpenAI"]
    end

    subgraph sdk_path["SDK path"]
        msgs["messages list<br/>system / user / assistant"]
        create["chat.completions.create"]
        msgs --> create
        sdk --> create
        create --> choice["choices[0].message.content<br/>+ usage / finish_reason"]
    end

    subgraph lc_path["LangChain path"]
        tmpl["ChatPromptTemplate"]
        chain["prompt | llm"]
        struct["optional<br/>with_structured_output"]
        tmpl --> chain
        llm --> chain
        chain --> aimsg["AIMessage.content"]
        chain --> struct
    end

    create --> api["OpenAI REST API"]
    chain --> api
```

### 4.2 Message shape mapping

```text
SDK messages                          LangChain ChatPromptTemplate
─────────────────────────────         ───────────────────────────────
{"role": "system", "content": ...}  →  ("system", "...")
{"role": "user", "content": ...}    →  ("human", "... {var} ...")
```

### 4.3 Recipes demonstrated below

| Path | Shape | When to use |
|------|-------|-------------|
| **SDK call** | `client.chat.completions.create(messages=...)` | Inspect `usage` / `finish_reason` |
| **LCEL text** | `ChatPromptTemplate \| llm` | Reusable prompt pipelines |
| **LCEL structured** | `prompt \| llm.with_structured_output(Schema)` | Typed extraction |

Setup, SDK calls, and chains are defined inline in the cells below — no shared helper modules required.


## 5. Local Python Examples


Build and validate `messages` locally before spending API credits. The same roles map one-to-one onto LangChain templates.


In [1]:
# Local Chat Completions message builder — no API required
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal

Role = Literal["system", "user", "assistant"]


@dataclass
class ChatTurn:
    role: Role
    content: str


@dataclass
class MessageBuilder:
    # Assemble OpenAI-style messages with basic validation.

    turns: list[ChatTurn] = field(default_factory=list)

    def system(self, content: str) -> "MessageBuilder":
        self.turns.append(ChatTurn("system", content.strip()))
        return self

    def user(self, content: str) -> "MessageBuilder":
        self.turns.append(ChatTurn("user", content.strip()))
        return self

    def assistant(self, content: str) -> "MessageBuilder":
        self.turns.append(ChatTurn("assistant", content.strip()))
        return self

    def build(self) -> list[dict[str, str]]:
        if not any(t.role == "user" for t in self.turns):
            raise ValueError("At least one user message is required.")
        return [{"role": t.role, "content": t.content} for t in self.turns]

    def as_langchain_tuples(self) -> list[tuple[str, str]]:
        # Map SDK roles to ChatPromptTemplate role names.
        role_map = {"system": "system", "user": "human", "assistant": "ai"}
        return [(role_map[t.role], t.content) for t in self.turns]


builder = (
    MessageBuilder()
    .system("You are a concise SRE assistant.")
    .user("Summarize why health checks should avoid live API calls.")
)
messages = builder.build()

print("SDK messages:")
for m in messages:
    preview = m["content"][:60].replace("\n", " ")
    print(f"  [{m['role']}] {preview}...")

print("\nLangChain-style tuples:")
for role, content in builder.as_langchain_tuples():
    preview = content[:60].replace("\n", " ")
    print(f"  ({role!r}, {preview!r}...)")
print(f"Approx chars: {sum(len(m['content']) for m in messages)}")


SDK messages:
  [system] You are a concise SRE assistant....
  [user] Summarize why health checks should avoid live API calls....

LangChain-style tuples:
  ('system', 'You are a concise SRE assistant.'...)
  ('human', 'Summarize why health checks should avoid live API calls.'...)
Approx chars: 88


## 6. OpenAI SDK Examples

```python
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(root / ".env")
client = OpenAI()
```


In [2]:
# Setup: load .env and create the OpenAI client
from __future__ import annotations

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import (
    APIConnectionError,
    AuthenticationError,
    OpenAI,
    RateLimitError,
)

root = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / ".env").is_file() or (p / "requirements.txt").is_file()
)
load_dotenv(root / ".env")

api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key.strip() or "your_openai_api_key" in api_key.lower():
    raise SystemExit("Set OPENAI_API_KEY in .env before running the API cells.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
client = OpenAI()
print("SDK client ready:", MODEL)


SDK client ready: gpt-4o-mini


In [3]:
# Connectivity check — Chat Completions + response metadata
messages = [
    {"role": "system", "content": "You are a concise assistant. Reply in one short sentence."},
    {"role": "user", "content": "Say: SDK connectivity OK."},
]

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0,
        max_tokens=20,
    )
    print("API call succeeded.")
    print(f"id    : {response.id}")
    print(f"model : {response.model}")
    print(f"reply : {response.choices[0].message.content}")
    print(f"finish: {response.choices[0].finish_reason}")
    if response.usage:
        print(
            f"tokens: prompt={response.usage.prompt_tokens}, "
            f"completion={response.usage.completion_tokens}"
        )
except AuthenticationError:
    print("Authentication failed. Check OPENAI_API_KEY.")
except RateLimitError:
    print("Rate limit or quota hit. Wait and retry.")
except APIConnectionError as exc:
    print(f"Network error: {exc}")


API call succeeded.
id    : chatcmpl-EAJ3pMMxKJ9RdNiMeewRPKSgr2opg
model : gpt-4o-mini-2024-07-18
reply : SDK connectivity OK.
finish: stop
tokens: prompt=29, completion=4


### 6.1 Inspect full response fields


In [4]:
# Print structured response metadata (usage, finish_reason, id)
messages = [
    {"role": "system", "content": "Reply with valid JSON only."},
    {"role": "user", "content": 'Return JSON: {"status": "ok", "items": ["a","b"]}'},
]

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.1,
        max_tokens=80,
    )
except (AuthenticationError, RateLimitError, APIConnectionError) as exc:
    print(f"Request failed: {type(exc).__name__}: {exc}")
else:
    choice = response.choices[0]
    print("--- choice ---")
    print(f"index         : {choice.index}")
    print(f"finish_reason : {choice.finish_reason}")
    print(f"role          : {choice.message.role}")
    print(f"content       : {choice.message.content}")
    print("--- usage ---")
    if response.usage:
        print(json.dumps(response.usage.model_dump(), indent=2))
    print(f"response.id   : {response.id}")


--- choice ---
index         : 0
finish_reason : stop
role          : assistant
content       : {
  "status": "ok",
  "items": [
    "a",
    "b"
  ]
}
--- usage ---
{
  "completion_tokens": 24,
  "prompt_tokens": 35,
  "total_tokens": 59,
  "completion_tokens_details": {
    "accepted_prediction_tokens": 0,
    "audio_tokens": 0,
    "reasoning_tokens": 0,
    "rejected_prediction_tokens": 0
  },
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cache_write_tokens": null,
    "cached_tokens": 0
  }
}
response.id   : chatcmpl-EAJ3qQFCbbhVOYzGfs0RLel2v6c0c


### 6.2 Streaming (SDK)


In [5]:
# Stream tokens as they arrive (SDK)
messages = [
    {"role": "system", "content": "Be brief."},
    {"role": "user", "content": "Name three HTTP status codes and one word each."},
]

try:
    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
        max_tokens=60,
        stream=True,
    )
    print("Stream: ", end="", flush=True)
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        print(delta, end="", flush=True)
    print()
except (AuthenticationError, RateLimitError, APIConnectionError) as exc:
    print(f"Stream failed: {type(exc).__name__}: {exc}")


Stream: 

1

.

200

 -

...

3

.

500

 -

 Error

## 7. LangChain Examples

Reuse the same `.env` and model id with LCEL for prompt templates, chains, and structured output.

```python
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv(root / ".env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
```


In [6]:
# Setup: ChatOpenAI using the same .env and MODEL
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

llm = ChatOpenAI(model=MODEL, temperature=0)
print("LangChain model ready:", llm.model_name)


LangChain model ready: gpt-4o-mini


In [7]:
# LCEL: ChatPromptTemplate | llm — same roles as the SDK messages above
sre_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a concise SRE assistant. Reply in 1–2 sentences."),
        ("human", "{question}"),
    ]
)

sre_chain = sre_prompt | llm
result = sre_chain.invoke(
    {"question": "Summarize why health checks should avoid live API calls."}
)

print("AIMessage.content:")
print(result.content)


AIMessage.content:
Health checks should avoid live API calls to prevent unnecessary load on the system, reduce the risk of cascading failures, and ensure that the health check itself does not become a point of failure. Instead, they should rely on internal metrics or status indicators that reflect the system's health without impacting performance.


In [8]:
# Structured output via with_structured_output (typed alternative to free-text JSON)
class HealthCheckTip(BaseModel):
    summary: str = Field(description="One-sentence summary")
    risk: Literal["low", "medium", "high"]
    avoid_live_calls: bool = Field(description="Whether live dependency calls should be avoided")


extract_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Extract structured guidance about health-check design. "
            "Use only the schema fields.",
        ),
        ("human", "Topic:\n<input>\n{topic}\n</input>"),
    ]
)

extract_llm = ChatOpenAI(model=MODEL, temperature=0.1).with_structured_output(HealthCheckTip)
extract_chain = extract_prompt | extract_llm

tip = extract_chain.invoke(
    {"topic": "Liveness probes that call a paid third-party API on every check."}
)

print("Parsed model:", tip)
print("As dict:", tip.model_dump())


Parsed model: summary='Avoid using liveness probes that rely on paid third-party APIs to prevent unnecessary costs and potential service disruptions.' risk='high' avoid_live_calls=True
As dict: {'summary': 'Avoid using liveness probes that rely on paid third-party APIs to prevent unnecessary costs and potential service disruptions.', 'risk': 'high', 'avoid_live_calls': True}


In [9]:
# Streaming via LCEL — chain.stream yields AIMessage chunks
stream_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Be brief."),
        ("human", "{question}"),
    ]
)
stream_chain = stream_prompt | llm

print("Stream: ", end="", flush=True)
for chunk in stream_chain.stream(
    {"question": "Name three HTTP status codes and one word each."}
):
    print(chunk.content or "", end="", flush=True)
print()


Stream: 

1

.

200

 -

...

3

.

500

 -

 Error

## 8. Implementation notes

1. **`load_dotenv` before clients** — Call `load_dotenv(root / ".env")` before `OpenAI()` or `ChatOpenAI`.
2. **Same `MODEL`** — Drive both SDK and LangChain from `OPENAI_MODEL` (default `gpt-4o-mini`).
3. **SDK for metadata** — Prefer `chat.completions.create` when you need `usage` / `finish_reason` / raw `id`.
4. **LCEL for composition** — Build `prompt | llm` once; reuse with `.invoke` / `.batch` / `.stream`.
5. **Role mapping** — SDK `user` maps to LangChain `human`; `assistant` maps to `ai`.
6. **`with_structured_output`** — Prefer over asking the SDK for "JSON only" when you need typed fields.


## 9. Best practices

- Never hardcode API keys; load from `.env` or a secret manager.
- Pin `model` in configuration; log `response.model` (SDK) or `llm.model_name` (LangChain).
- Set `temperature=0` (or low) for structured extraction; raise slightly for creative drafts.
- Log `usage` on SDK calls for cost dashboards; use LangChain callbacks when equivalent telemetry is needed in chains.
- Prefer `ChatPromptTemplate | llm` in application code once prompts stabilize.
- Validate structured results with Pydantic / `Literal` — do not trust free-text JSON alone.


## 10. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `AuthenticationError` | Missing, placeholder, or revoked key | Set a valid `OPENAI_API_KEY` in the project-root `.env` |
| `RateLimitError` | Quota, concurrency, or TPM limits | Back off, batch requests, or upgrade the plan |
| `APIConnectionError` | Network, proxy, or DNS | Check connectivity; retry with a timeout |
| Empty `content` | Refusal, filter, or tool-call response | Inspect `finish_reason` and the message object |
| `ModuleNotFoundError: openai` / `langchain_openai` | Wrong interpreter or environment | Install dependencies in the active environment |
| Key works in terminal, not notebook | cwd / `.env` path mismatch | Resolve the project root and load `.env` explicitly |
| JSON with prose wrapper | Free-text JSON request via SDK | Switch to `with_structured_output` |
| Unexpected cost growth | No `max_tokens`, oversized prompts | Cap tokens; estimate with `tiktoken` before large runs |


## 11. Validation checklist

1. Run the message builder; confirm SDK messages and LangChain tuples both print.
2. Run the SDK connectivity check; confirm `id`, `model`, `finish_reason`, and reply print.
3. Run response inspection; confirm `usage` fields appear.
4. Run `sre_chain.invoke`; confirm `AIMessage.content` is non-empty.
5. Run structured extraction; confirm a `HealthCheckTip` instance with a valid `risk`.
6. Optionally compare SDK `stream=True` with `chain.stream` output.


## 12. Summary

- Bootstrap with `load_dotenv(root / ".env")`, then `OpenAI()` and/or `ChatOpenAI`.
- SDK Chat Completions take a `messages` list; LangChain uses `ChatPromptTemplate | llm`.
- Parse `choices[0].message.content`, `finish_reason`, and `usage` on the SDK path.
- Prefer LCEL with `with_structured_output` for reusable prompts and typed extraction.
